In [1]:
import tarfile
import io
from glob import glob
import os
from pathlib import Path
import polars as pl

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR = "/cbioportal_tabular_downloads"  # folder with all .tar.gz files
SKIP_ROWS = 4  # cBioPortal metadata header rows to skip
TARGET_FILES = {"data_clinical_patient.txt", "data_clinical_sample.txt"}

In [2]:
def read_clinical_from_tar(tar_path: str) -> dict[str, pl.DataFrame | None]:
    """
    Opens a .tar.gz in-memory and returns a dict:
        {"patient": <DataFrame or None>, "sample": <DataFrame or None>}
    Never writes anything to disk.
    """
    results = {"patient": None, "sample": None}
    try:
        with tarfile.open(tar_path, "r:gz") as tar:
            for member in tar.getmembers():
                fname = Path(member.name).name  # basename only
                if fname not in TARGET_FILES:
                    continue
                f = tar.extractfile(member)
                if f is None:
                    continue
                raw = f.read()
                # Skip the 4 metadata comment lines then parse
                lines = raw.decode("utf-8", errors="replace").splitlines()
                data_lines = "\n".join(lines[SKIP_ROWS:]).encode("utf-8")
                try:
                    df = pl.read_csv(
                        io.BytesIO(data_lines),
                        separator="\t",
                        infer_schema_length=500,
                        ignore_errors=True,
                    )
                    key = "patient" if "patient" in fname else "sample"
                    results[key] = df
                except Exception as e:
                    print(f"  [WARN] Could not parse {fname} in {tar_path}: {e}")
    except Exception as e:
        print(f"  [ERROR] Could not open {tar_path}: {e}")
    return results

In [3]:
tar_files = sorted(glob(os.path.join(DATA_DIR, "*.tar.gz")))
print(tar_files)
print(f"Found {len(tar_files)} archives.\n")

summary_rows = []  # one row per archive
all_patient_dfs = []
all_sample_dfs = []

for i, tar_path in enumerate(tar_files, 1):
    study_id = Path(tar_path).stem.replace(".tar", "")
    print(f"[{i:>3}/{len(tar_files)}] {study_id}", end=" ... ", flush=True)

    dfs = read_clinical_from_tar(tar_path)
    p, s = dfs["patient"], dfs["sample"]

    p_rows = len(p) if p is not None else 0
    s_rows = len(s) if s is not None else 0
    p_cols = len(p.columns) if p is not None else 0
    s_cols = len(s.columns) if s is not None else 0
    print(f"patient={p_rows} rows | sample={s_rows} rows")

    summary_rows.append(
        {
            "study_id": study_id,
            "patient_rows": p_rows,
            "patient_cols": p_cols,
            "sample_rows": s_rows,
            "sample_cols": s_cols,
            "has_patient": p is not None,
            "has_sample": s is not None,
        }
    )

    # Tag with study id before stacking
    if p is not None:
        all_patient_dfs.append(p.with_columns(pl.lit(study_id).alias("study_id")))
    if s is not None:
        all_sample_dfs.append(s.with_columns(pl.lit(study_id).alias("study_id")))

['/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/downloads_valid/acc_tcga.tar.gz', '/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/downloads_valid/acc_tcga_gdc.tar.gz', '/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/downloads_valid/acc_tcga_pan_can_atlas_2018.tar.gz', '/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/downloads_valid/aml_tcga_gdc.tar.gz', '/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/downloads_valid/blca_msk_tcga_2020.tar.gz', '/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/downloads_valid/blca_tcga.tar.gz', '/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/downloads_valid/blca_tcga_gdc.tar.gz', '/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/downloads_valid/blca_tcga_pan_can_atlas_2018.tar.gz', '/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/downloads_valid/blca_tcga_pub.tar.gz', '/home/youssef_mohamma

In [4]:
summary = pl.DataFrame(summary_rows)

print("\n" + "=" * 60)
print("SUMMARY TABLE (all studies)")
print("=" * 60)
print(summary)

print("\n── Top 10 studies by patient count ──")
print(
    summary.sort("patient_rows", descending=True)
    .head(10)
    .select(["study_id", "patient_rows", "sample_rows"])
)

print("\n── Studies missing patient file ──")
missing_p = summary.filter(~pl.col("has_patient"))
print(missing_p.select("study_id") if len(missing_p) else "None")

print("\n── Studies missing sample file ──")
missing_s = summary.filter(~pl.col("has_sample"))
print(missing_s.select("study_id") if len(missing_s) else "None")

# ── Aggregate numeric stats across ALL patient files ─────────────────────────
if all_patient_dfs:
    print("\n" + "=" * 60)
    print("POOLED PATIENT EDA")
    print("=" * 60)

    # Diagonal stack (union_frames handles mismatched schemas gracefully)
    pooled_patient = pl.concat(all_patient_dfs, how="diagonal_relaxed")
    print(f"Total pooled patient rows : {len(pooled_patient):,}")
    print(f"Total pooled patient cols : {len(pooled_patient.columns)}")

    # Numeric column stats
    num_cols = [
        c
        for c, t in zip(pooled_patient.columns, pooled_patient.dtypes)
        if t in (pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Int16, pl.Int8)
    ]
    if num_cols:
        print("\nNumeric column statistics:")
        print(pooled_patient.select(num_cols).describe())

    # Common string columns of interest
    for col in ["SEX", "RACE", "ETHNICITY", "OS_STATUS", "CANCER_TYPE"]:
        if col in pooled_patient.columns:
            vc = (
                pooled_patient[col]
                .drop_nulls()
                .value_counts()
                .sort("count", descending=True)
                .head(10)
            )
            print(f"\nValue counts — {col}:")
            print(vc)

    # Save pooled summary
    summary.write_csv("eda_study_summary.csv")
    print("\n[Saved] eda_study_summary.csv")

if all_sample_dfs:
    print("\n" + "=" * 60)
    print("POOLED SAMPLE EDA")
    print("=" * 60)
    pooled_sample = pl.concat(all_sample_dfs, how="diagonal_relaxed")
    print(f"Total pooled sample rows  : {len(pooled_sample):,}")
    print(f"Total pooled sample cols  : {len(pooled_sample.columns)}")

    num_cols_s = [
        c
        for c, t in zip(pooled_sample.columns, pooled_sample.dtypes)
        if t in (pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Int16, pl.Int8)
    ]
    if num_cols_s:
        print("\nNumeric column statistics:")
        print(pooled_sample.select(num_cols_s).describe())

    for col in ["SAMPLE_TYPE", "ONCOTREE_CODE", "CANCER_TYPE", "TMB_NONSYNONYMOUS"]:
        if col in pooled_sample.columns:
            if pooled_sample[col].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32):
                desc = pooled_sample.select(pl.col(col).drop_nulls().describe())
                print(f"\nDistribution — {col}:")
                print(desc)
            else:
                vc = (
                    pooled_sample[col]
                    .drop_nulls()
                    .value_counts()
                    .sort("count", descending=True)
                    .head(10)
                )
                print(f"\nValue counts — {col}:")
                print(vc)

print("\nDone ✓")


SUMMARY TABLE (all studies)
shape: (123, 7)
┌──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬────────────┐
│ study_id     ┆ patient_rows ┆ patient_col ┆ sample_rows ┆ sample_cols ┆ has_patient ┆ has_sample │
│ ---          ┆ ---          ┆ s           ┆ ---         ┆ ---         ┆ ---         ┆ ---        │
│ str          ┆ i64          ┆ ---         ┆ i64         ┆ i64         ┆ bool        ┆ bool       │
│              ┆              ┆ i64         ┆             ┆             ┆             ┆            │
╞══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪════════════╡
│ acc_tcga     ┆ 92           ┆ 82          ┆ 92          ┆ 27          ┆ true        ┆ true       │
│ acc_tcga_gdc ┆ 92           ┆ 30          ┆ 92          ┆ 10          ┆ true        ┆ true       │
│ acc_tcga_pan ┆ 92           ┆ 38          ┆ 92          ┆ 19          ┆ true        ┆ true       │
│ _can_atlas_2 ┆              ┆             ┆ 